In [1]:
import pandas as pd
from pathlib import Path


def read_parquet_dataset(path: Path) -> pd.DataFrame:
    parquet_files = sorted(path.rglob("*.parquet"))
    dfs = [pd.read_parquet(p) for p in parquet_files]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

root = Path(".")

customers = read_parquet_dataset(root / "churn_customers")
orders = read_parquet_dataset(root / "churn_orders")
product_usage = read_parquet_dataset(root / "churn_product_usage")
subscriptions = read_parquet_dataset(root / "churn_subscriptions")
support_tickets = read_parquet_dataset(root / "churn_support_tickets")
marketing_interactions = read_parquet_dataset(root / "churn_marketing_interactions")
payments = read_parquet_dataset(root / "churn_payments")

for name, df in [
    ("customers", customers),
    ("orders", orders),
    ("product_usage", product_usage),
    ("subscriptions", subscriptions),
    ("support_tickets", support_tickets),
    ("marketing_interactions", marketing_interactions),
    ("payments", payments),
]:
    print("="*60)
    print(name)
    print(df.shape)
    print(df.dtypes)

customers
(10002, 9)
customer_id                int32
gender                       str
birth_date        datetime64[ns]
region                       str
city                         str
signup_date       datetime64[ns]
account_status               str
closed_date       datetime64[ns]
last_login_at     datetime64[ns]
dtype: object
orders
(179204, 6)
order_id                   int32
customer_id                int32
order_date        datetime64[ns]
status                       str
total_amount             float32
payment_method               str
dtype: object
product_usage
(770082, 6)
usage_id                         int32
customer_id                      int32
event_date              datetime64[ns]
event_type                         str
session_duration_sec             int32
device                             str
dtype: object
subscriptions
(12615, 8)
subscription_id             int32
customer_id                 int32
plan_tier                     str
change_type                   str
st

In [2]:
for name, df in [
    ("customers", customers),
    ("orders", orders),
    ("product_usage", product_usage),
    ("subscriptions", subscriptions),
    ("support_tickets", support_tickets),
    ("marketing_interactions", marketing_interactions),
    ("payments", payments),
]:
    print("="*60)
    print(name)
    print(df.head())

customers
   customer_id gender birth_date    region         city signup_date  \
0            1      M 1991-05-19  Mien Nam  Ho Chi Minh  2025-03-24   
1            2      M 1979-12-31  Mien Bac   Quang Ninh  2025-05-15   
2            3      M 1987-07-07  Mien Nam      Can Tho  2025-12-16   
3            4      F 1978-12-13  Mien Nam      Can Tho  2025-03-23   
4            5      F 1990-08-13  Mien Bac       Ha Noi  2026-04-14   

  account_status closed_date       last_login_at  
0         Active         NaT 2026-07-26 09:27:00  
1         Active         NaT 2025-11-21 03:19:00  
2         Active         NaT 2026-07-02 17:00:00  
3         Active         NaT 2026-07-17 15:02:00  
4         Active         NaT 2026-06-17 16:09:00  
orders
   order_id  customer_id          order_date     status  total_amount  \
0       172           23 2023-10-18 15:00:00   Returned     1496000.0   
1       214           26 2023-10-22 08:00:00  Completed     4910000.0   
2       215           26 2023-1

In [3]:
customers.dtypes

customer_id                int32
gender                       str
birth_date        datetime64[ns]
region                       str
city                         str
signup_date       datetime64[ns]
account_status               str
closed_date       datetime64[ns]
last_login_at     datetime64[ns]
dtype: object

In [4]:
#support_tickets.dtypes
support_tickets["created_at"] = pd.to_datetime(support_tickets["created_at"])
support_tickets.head()


,ticket_id,customer_id,created_at,category,priority,resolution_hours,csat_score
0,188,73,2023-10-06 04:00:00,Other,Medium,4.3,3.0
1,273,111,2023-10-25 02:00:00,Shipping,Low,5.5,NaN
2,543,220,2023-10-19 01:00:00,Account,Medium,2.6,4.0
3,944,402,2023-10-03 10:00:00,Shipping,High,1.5,2.0
4,1250,549,2023-10-29 20:00:00,Product,Medium,11.2,2.0


In [5]:
marketing_interactions.dtypes
marketing_interactions["sent_at"] = pd.to_datetime(marketing_interactions["sent_at"])
marketing_interactions.dtypes

interaction_id             int32
customer_id                int32
sent_at           datetime64[ns]
opened                      bool
clicked                     bool
converted                   bool
channel                      str
dtype: object

In [6]:
#payments.dtypes
payments["payment_date"] = pd.to_datetime(payments["payment_date"])
payments.dtypes

payment_id                  int32
customer_id                 int32
order_id                  float64
subscription_id           float64
payment_date       datetime64[ns]
amount                    float32
status                        str
method                        str
dtype: object

In [7]:
#orders.dtypes
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders.head()

,order_id,customer_id,order_date,status,total_amount,payment_method
0,172,23,2023-10-18 15:00:00,Returned,1496000.0,COD
1,214,26,2023-10-22 08:00:00,Completed,4910000.0,BankTransfer
2,215,26,2023-10-11 18:00:00,Completed,1190000.0,BankTransfer
3,262,27,2023-10-11 12:00:00,Completed,5246000.0,BankTransfer
4,372,32,2023-10-10 17:00:00,Cancelled,1468000.0,CreditCard


In [8]:
snapshot_date = pd.to_datetime("2026-01-01")
lookback_days = 60
lookup_days = 30
before_date = snapshot_date - pd.Timedelta(days=lookback_days)
after_date = snapshot_date + pd.Timedelta(days=lookup_days)
print(snapshot_date)
print(before_date)
print(after_date)

2026-01-01 00:00:00
2025-11-02 00:00:00
2026-01-31 00:00:00


In [9]:
"""
product_usage
subscriptions
customers
support_tickets
marketing_interactions
payments
orders
"""
before_snapshot = {
    "customers": customers[
        (customers["signup_date"] < snapshot_date)
    ].copy(),
    "product_usage": product_usage[
        (product_usage["event_date"] >= before_date) &
        (product_usage["event_date"] < snapshot_date)
    ].copy(),
    "subscriptions": subscriptions[
        (subscriptions["start_date"] < snapshot_date) &
        ((subscriptions["end_date"] >= before_date) | (subscriptions["end_date"].isna()))
    ].copy(),
    "support_tickets": support_tickets[
        (support_tickets["created_at"] >= before_date) &
        (support_tickets["created_at"] < snapshot_date)
    ].copy(),
    "marketing_interactions": marketing_interactions[
        (marketing_interactions["sent_at"] >= before_date) &
        (marketing_interactions["sent_at"] < snapshot_date)
    ].copy(),
    "payments": payments[
        (payments["payment_date"] >= before_date) &
        (payments["payment_date"] < snapshot_date)
    ].copy(),
    "orders": orders[
        (orders["order_date"] >= before_date) &
        (orders["order_date"] < snapshot_date)
    ].copy()
}

In [10]:
after_snapshot = {
    "customers": customers[
        (customers["signup_date"] < snapshot_date)
    ].copy(),
    "product_usage": product_usage[
        (product_usage["event_date"] >= snapshot_date) &
        (product_usage["event_date"] < after_date)
    ].copy(),
    "subscriptions": subscriptions[
        (subscriptions["start_date"] >= snapshot_date) &
        (subscriptions["start_date"] < after_date)
    ].copy(),
    "support_tickets": support_tickets[
        (support_tickets["created_at"] >= snapshot_date) &
        (support_tickets["created_at"] < after_date)
    ].copy(),
    "marketing_interactions": marketing_interactions[
        (marketing_interactions["sent_at"] >= snapshot_date) &
        (marketing_interactions["sent_at"] < after_date)
    ].copy(),
    "payments": payments[
        (payments["payment_date"] >= snapshot_date) &
        (payments["payment_date"] < after_date)
    ].copy(),
    "orders": orders[
        (orders["order_date"] >= snapshot_date) &
        (orders["order_date"] < after_date)
    ].copy()
}

In [11]:
"""
product_usage
subscriptions
customers
support_tickets
marketing_interactions
payments
orders
"""
product_usage.dtypes
feature_product_usage = before_snapshot["product_usage"].groupby("customer_id").agg(
    total_usages=("usage_id", "count"),
    avg_session_duration=("session_duration_sec", "mean")
).reset_index()
del product_usage

before_snapshot["subscriptions"].sort_values(["customer_id", "start_date"])
feature_subscriptions = before_snapshot["subscriptions"].groupby("customer_id").agg(
    total_subscriptions=("subscription_id", "count"),
    current_auto_renew=("auto_renew", "last"),
    current_sub_plan=("plan_tier", "last"),
    current_sub_status=("status", "last")
).reset_index()
feature_subscriptions.head()
del subscriptions

In [12]:
feature_customers = before_snapshot["customers"].reset_index()
feature_customers.head()

,index,customer_id,gender,birth_date,region,city,signup_date,account_status,closed_date,last_login_at
0,0,1,M,1991-05-19,Mien Nam,Ho Chi Minh,2025-03-24,Active,NaT,2026-07-26 09:27:00
1,1,2,M,1979-12-31,Mien Bac,Quang Ninh,2025-05-15,Active,NaT,2025-11-21 03:19:00
2,2,3,M,1987-07-07,Mien Nam,Can Tho,2025-12-16,Active,NaT,2026-07-02 17:00:00
3,3,4,F,1978-12-13,Mien Nam,Can Tho,2025-03-23,Active,NaT,2026-07-17 15:02:00
4,5,6,F,2006-06-24,Mien Trung,Nha Trang,2025-01-08,Active,NaT,2026-07-06 04:26:00


In [13]:
feature_support_tickets = before_snapshot["support_tickets"].groupby("customer_id").agg(
    total_tickets=("ticket_id", "count"),
    avg_ticket_resolution=("resolution_hours", "mean"),
    avg_csat_score=("csat_score", "mean"),
    count_null_csat_score=("csat_score", lambda x: x.isna().sum()),
).reset_index()
feature_support_tickets

,customer_id,total_tickets,avg_ticket_resolution,avg_csat_score,count_null_csat_score
0,19,1,7.60,NaN,1
1,20,1,11.80,4.0,0
2,21,1,6.90,NaN,1
3,33,2,7.65,3.5,0
4,38,2,3.80,3.0,0
...,...,...,...,...,...
1643,9968,1,8.10,4.0,0
1644,9979,1,3.80,NaN,1
1645,9983,1,4.80,4.0,0
1646,9996,1,2.90,NaN,1


In [14]:
feature_marketing_interactions = before_snapshot["marketing_interactions"].groupby("customer_id").agg(
    total_marketings=("interaction_id", "count"),
    total_mi_opened=("opened", "sum"),
    total_mi_clicked=("clicked", "sum"),
    total_mi_converted=("converted", "sum")
)
feature_marketing_interactions

,total_marketings,total_mi_opened,total_mi_clicked,total_mi_converted
customer_id,,,,
1,2,0,0,0
2,3,2,1,0
3,1,0,0,0
4,3,0,0,0
6,6,2,1,0
...,...,...,...,...
9996,2,1,1,0
9997,2,0,0,0
9998,1,0,0,0


In [15]:
feature_payments = before_snapshot["payments"].groupby("customer_id").agg(
    total_payments=("payment_id", "count"),
    count_payment_failed=("status", lambda x: (x == "Failed").sum()),
    avg_payment_amount=("amount", "mean")
)
feature_payments


,total_payments,count_payment_failed,avg_payment_amount
customer_id,,,
1,2,0,99000.000
4,2,0,1644000.000
6,2,0,99000.000
8,5,0,878000.000
9,4,0,323250.000
...,...,...,...
9996,6,2,1221833.375
9997,4,0,611500.000
9998,4,0,989250.000


In [16]:
feature_orders = before_snapshot["orders"].groupby("customer_id").agg(
    total_orders=("order_id", "count"),
    count_order_completed=("status", lambda x: (x == "Completed").sum()),
    avg_order_amount=("total_amount", "mean")
).reset_index()
feature_orders

,customer_id,total_orders,count_order_completed,avg_order_amount
0,4,3,3,1.937000e+06
1,8,7,6,1.490429e+06
2,9,3,3,3.503333e+05
3,13,4,2,1.122500e+06
4,17,3,3,7.313333e+05
...,...,...,...,...
4563,9994,2,2,2.360000e+05
4564,9996,5,4,1.484000e+06
4565,9997,3,3,1.017667e+06
4566,9998,7,7,9.600000e+05


In [17]:
#del dataset
dataset = feature_customers.copy()

In [18]:
dataset = dataset.merge(
    feature_marketing_interactions,
    on="customer_id",
    how="left"
)
dataset = dataset.merge(
    feature_orders,
    on="customer_id",
    how="left"
)
dataset = dataset.merge(
    feature_payments,
    on="customer_id",
    how="left"
)
dataset = dataset.merge(
    feature_support_tickets,
    on="customer_id",
    how="left"
)
dataset = dataset.merge(
    feature_product_usage,
    on="customer_id",
    how="left"
)
dataset = dataset.merge(
    feature_subscriptions,
    on="customer_id",
    how="left"
)
dataset.columns

Index(['index', 'customer_id', 'gender', 'birth_date', 'region', 'city',
       'signup_date', 'account_status', 'closed_date', 'last_login_at',
       'total_marketings', 'total_mi_opened', 'total_mi_clicked',
       'total_mi_converted', 'total_orders', 'count_order_completed',
       'avg_order_amount', 'total_payments', 'count_payment_failed',
       'avg_payment_amount', 'total_tickets', 'avg_ticket_resolution',
       'avg_csat_score', 'count_null_csat_score', 'total_usages',
       'avg_session_duration', 'total_subscriptions', 'current_auto_renew',
       'current_sub_plan', 'current_sub_status'],
      dtype='str')

In [19]:
dataset.dtypes

index                             int64
customer_id                       int32
gender                              str
birth_date               datetime64[ns]
region                              str
city                                str
signup_date              datetime64[ns]
account_status                      str
closed_date              datetime64[ns]
last_login_at            datetime64[ns]
total_marketings                float64
total_mi_opened                 float64
total_mi_clicked                float64
total_mi_converted              float64
total_orders                    float64
count_order_completed           float64
avg_order_amount                float32
total_payments                  float64
count_payment_failed            float64
avg_payment_amount              float32
total_tickets                   float64
avg_ticket_resolution           float32
avg_csat_score                  float64
count_null_csat_score           float64
total_usages                    float64


In [20]:
dataset["age"] = ((snapshot_date - dataset["birth_date"]).dt.days/365.25).round(1)
dataset.drop(columns = ["birth_date"], inplace=True)
dataset["customer_tenure"] = ((snapshot_date - dataset["signup_date"]).dt.days/365.25).round(1)
dataset.drop(columns = ["signup_date"], inplace = True)
dataset.head()

,index,customer_id,gender,region,city,account_status,closed_date,last_login_at,total_marketings,total_mi_opened,...,avg_csat_score,count_null_csat_score,total_usages,avg_session_duration,total_subscriptions,current_auto_renew,current_sub_plan,current_sub_status,age,customer_tenure
0,0,1,M,Mien Nam,Ho Chi Minh,Active,NaT,2026-07-26 09:27:00,2.0,0.0,...,NaN,NaN,NaN,NaN,1.0,True,Plus,Active,34.6,0.8
1,1,2,M,Mien Bac,Quang Ninh,Active,NaT,2025-11-21 03:19:00,3.0,2.0,...,NaN,NaN,6.0,800.833333,1.0,False,Plus,Expired,46.0,0.6
2,2,3,M,Mien Nam,Can Tho,Active,NaT,2026-07-02 17:00:00,1.0,0.0,...,NaN,NaN,NaN,NaN,1.0,True,Plus,Active,38.5,0.0
3,3,4,F,Mien Nam,Can Tho,Active,NaT,2026-07-17 15:02:00,3.0,0.0,...,NaN,NaN,NaN,NaN,1.0,True,Free,Active,47.1,0.8
4,5,6,F,Mien Trung,Nha Trang,Active,NaT,2026-07-06 04:26:00,6.0,2.0,...,NaN,NaN,NaN,NaN,1.0,True,Plus,Active,19.5,1.0


In [21]:
dataset.to_csv("feature_dataset.csv", index=False)